# 01 Dataset exploration (IDD Detection)
Run after `scripts/prepare_dataset.py`. All numbers come from the converted dataset; nothing is typed in by hand.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from IPython.display import Image, display
from src.data.config import load_classes, load_yaml, resolve
from src.visualization.eda import load_boxes, compute_stats, run_eda

cfg = load_yaml(resolve('configs/dataset.yaml'))
classes = load_classes(resolve(cfg['class_mapping']))
names = [c.name for c in classes]
groups = {c.name: c.group for c in classes}
yolo = resolve(cfg['yolo']['root'])
manifest = pd.read_csv(yolo / 'manifest.csv')
boxes = load_boxes(yolo, names)
manifest.groupby('split').size()

## Conversion report and issues

In [ ]:
print(json.dumps(json.load(open(yolo / 'conversion_report.json')), indent=2))
issues = pd.read_csv(yolo / 'conversion_issues.csv')
issues.groupby(['severity', 'code']).size()

## Class distribution

In [ ]:
counts = boxes.groupby(['class', 'split']).size().unstack(fill_value=0).reindex(names, fill_value=0)
counts['group'] = counts.index.map(groups)
counts

## Statistics and figures (written to results/eda/)

In [ ]:
out = resolve('results/eda')
stats = run_eda(yolo, names, groups, out, cfg['release']['name'])
for f in stats['figures']:
    display(Image(filename=str(out / f)))

## Observations
Write these after looking at the real figures (imbalance, rare classes, small objects, train vs val differences).